### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [8]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='The short answer is that parrots don’t actually "talk" in the human sense. They are **mimicking sounds** they hear, driven by a combination of biological capability, social instinct, and reinforcement.\n\nHere’s a breakdown of why and how parrots learn to vocalize human speech:\n\n### 1. Biological Capability: The Siropharyngeal Air Sac\nParrots have a unique vocal anatomy that allows them to produce a wide range of sounds:\n- **Siropharyngeal Air Sac**: This structure at the base of the throat acts like a bellows, allowing them to produce complex, resonant sounds.\n- **Tongue and Beak Control**: Parrots have a flexible, muscular tongue and a beak that can shape sounds with precision, similar to how humans use their lips, tongue, and teeth.\n- **Lack of a Larynx (Voice Box)**: Unlike mammals, parrots don’t have a larynx. Instead, they produce sound by forcing air past their syrinx (a vocal organ in birds), which vibrates membranes to create sound. This allows for gre

In [9]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [10]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print("Response: ",response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Response:  content='' additional_kwargs={'tool_calls': [{'id': '5bagq6dgc', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.073325502, 'completion_tokens_details': None, 'prompt_time': 0.01966997, 'prompt_tokens_details': None, 'queue_time': 0.773991217, 'total_time': 0.092995472}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1ce2e-7e1f-77d7-92f7-a83f811e46d1-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '5bagq6dgc', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [12]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print("Message: ",final_response)
print("Result: ",final_response.text)
# "The current weather in Boston is 72°F and sunny."

Message:  content="It's sunny in Boston! ☀️" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 325, 'total_tokens': 335, 'completion_time': 0.019402453, 'completion_tokens_details': None, 'prompt_time': 0.024693373, 'prompt_tokens_details': None, 'queue_time': 0.051420487, 'total_time': 0.044095826}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1ce30-50af-74dc-a648-831f7602625d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 325, 'output_tokens': 10, 'total_tokens': 335}
Result:  It's sunny in Boston! ☀️
